# 1.3 — Ingesta: Índice de Marginación (CONAPO/INEGI)

**Capa 1 del protocolo.** Template para descargar, limpiar y validar el índice
de marginación en dos granularidades:

- **AGEB urbana 2020** (IMU, CONAPO) — análisis intra-urbano de Hermosillo. *(pendiente)*
- **Municipal** (IMM 2015/2020, CONAPO) — contexto territorial y serie temporal. *(implementado en este notebook)*

> Nota de CONAPO: las estimaciones 2010 y 2015 con el método DP2 son una referencia para
> las de 2020 y no reemplazan oficialmente a las publicadas en su momento. Solo la serie
> 2010-2015-2020 usa el mismo método y es comparable entre sí.

> Alcance temporal: el panel del proyecto cubre 2015-2020 (`PANEL_START_YEAR` y
> `PANEL_END_YEAR` en `config.py`). La marginación se publica solo en años censales
> (2010, 2015, 2020), así que aquí se usan 2015 y 2020; `panel.py` interpola entre ambos
> para tener un valor mensual. El año 2010 queda fuera de la ventana.

## Entradas

| Ruta | Contenido |
|------|-----------|
| `data/raw/marginacion_municipal_imm_{2015,2020}.xlsx` | IMM municipal nacional (Excel de CONAPO, se descarga aquí) |
| `data/raw/imu_2020.csv` | IMU 2020 nacional por AGEB (CONAPO) — pendiente |
| `data/external/` | Cartografía AGEB/municipio INEGI MGN 2020 — pendiente |

## Salidas

- `data/interim/marginacion/marginacion_municipal_{2015,2020}.csv` — municipios de Sonora.
- `data/interim/marginacion/marginacion_imu_2020.csv` — AGEBs de Hermosillo (pendiente).

## Checklist

- [x] Descargar las fuentes municipales.
- [x] Filtrar Sonora por clave de entidad (26).
- [x] Unificar nombres de columnas entre años.
- [x] Guardar en `data/interim/marginacion/`.
- [ ] AGEB urbana 2020 (IMU) y su validación con `MarginacionAgebSchema`.
- [ ] Cartografía AGEB.

## Configuración e imports

In [4]:
from pathlib import Path
import re
import sys
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

ROOT_DIR = Path.cwd().parent.parent  # notebooks/<seccion>/ -> raiz del proyecto
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))


from sonora_health_climate_eda.config import (
    HERMOSILLO_CVE_ENT,
    HERMOSILLO_CVE_MUN,
    INTERIM_DATA_DIR,
    PANEL_END_YEAR,
    PANEL_START_YEAR,
    RAW_DATA_DIR,
)

MARGINACION_INTERIM_DIR = INTERIM_DATA_DIR / "marginacion"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
MARGINACION_INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# Fuentes municipales de CONAPO (Excel por año censal)
URLS_IMM_TODOS = {
    2010: "https://conapo.segob.gob.mx/work/models/CONAPO/Datos_Abiertos/Municipio/IMM_DP2_2010.xlsx",
    2015: "https://conapo.segob.gob.mx/work/models/CONAPO/Datos_Abiertos/Municipio/IMM_DP2_2015.xlsx",
    2020: "https://conapo.segob.gob.mx/work/models/CONAPO/Datos_Abiertos/Municipio/IMM_2020.xlsx",
}

# Solo los años censales dentro de la ventana del panel (config.py: 2015-2020)
ANIOS_MARGINACION = [a for a in URLS_IMM_TODOS if PANEL_START_YEAR <= a <= PANEL_END_YEAR]
URLS_IMM = {a: URLS_IMM_TODOS[a] for a in ANIOS_MARGINACION}

# Claves INEGI (vienen de config.py): Sonora = 26; Hermosillo = 26030
CVE_ENT_SONORA = int(HERMOSILLO_CVE_ENT)
CVE_MUN_HERMOSILLO = int(HERMOSILLO_CVE_MUN)

print("Años de marginación:", ANIOS_MARGINACION)
print("Raw:", RAW_DATA_DIR)
print("Salida:", MARGINACION_INTERIM_DIR)

Años de marginación: [2015, 2020]
Raw: /Users/benancio/Documents/sonora-health-climate-eda/data/raw
Salida: /Users/benancio/Documents/sonora-health-climate-eda/data/interim/marginacion


## Descarga

`descargar_archivo` baja el archivo por bloques a un `.part`, verifica que sea un Excel real
(a veces estos sitios devuelven una página HTML de error con código 200) y solo entonces
lo renombra. Si el archivo ya existe, se omite.

**Plan B:** si el servidor de CONAPO bloquea la descarga automática, bájalos a mano desde
la página de CONAPO (se recomienda Firefox o Edge) y guárdalos en `data/raw/` con los nombres
`marginacion_municipal_imm_2015.xlsx` y `marginacion_municipal_imm_2020.xlsx`. El notebook los tomará solos.

In [5]:
FIRMA_XLSX = b"PK"                                   # .xlsx es un zip
FIRMA_XLS = b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1"      # .xls antiguo (OLE2)


def crear_sesion() -> requests.Session:
    """Sesión HTTP con User-Agent de navegador y reintentos automáticos."""
    sesion = requests.Session()
    sesion.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
        ),
        "Accept": "*/*",
    })
    reintentos = Retry(total=4, backoff_factor=1.5,
                       status_forcelist=(429, 500, 502, 503, 504),
                       allowed_methods=("GET",))
    sesion.mount("https://", HTTPAdapter(max_retries=reintentos))
    sesion.mount("http://", HTTPAdapter(max_retries=reintentos))
    return sesion


def descargar_archivo(url: str, destino: Path, sesion: requests.Session,
                      sobrescribir: bool = False) -> Path:
    """Descarga `url` a `destino` por bloques y verifica que sea un Excel real."""
    if destino.exists() and not sobrescribir:
        print(f"Ya existe, se omite: {destino.name}")
        return destino

    temporal = destino.with_suffix(destino.suffix + ".part")
    with sesion.get(url, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        with open(temporal, "wb") as f:
            for bloque in resp.iter_content(chunk_size=1024 * 256):
                f.write(bloque)

    with open(temporal, "rb") as f:
        inicio = f.read(8)
    if not (inicio.startswith(FIRMA_XLSX) or inicio.startswith(FIRMA_XLS)):
        temporal.unlink()
        raise ValueError("Lo descargado no parece un Excel (¿página de error o bloqueo?).")

    temporal.replace(destino)
    print(f"Descargado: {destino.name} ({destino.stat().st_size / 1024:.1f} KB)")
    return destino


sesion = crear_sesion()
RUTAS_IMM = {anio: RAW_DATA_DIR / f"marginacion_municipal_imm_{anio}.xlsx" for anio in URLS_IMM}

for anio, url in URLS_IMM.items():
    try:
        descargar_archivo(url, RUTAS_IMM[anio], sesion)
    except Exception as e:
        print(f"⚠ No se pudo descargar {anio}: {e}")
        print(f"   Bájalo a mano y guárdalo como {RUTAS_IMM[anio].name} en {RAW_DATA_DIR}")

faltan = [a for a, r in RUTAS_IMM.items() if not r.exists()]
if faltan:
    raise FileNotFoundError(f"Faltan los archivos de los años {faltan} en {RAW_DATA_DIR}")

# AGEB urbana 2020: pendiente (los comandos están pausados en dataset.py)
# from sonora_health_climate_eda.dataset import download_marginacion_ageb_data
# download_marginacion_ageb_data()

Descargado: marginacion_municipal_imm_2015.xlsx (1020.0 KB)
Descargado: marginacion_municipal_imm_2020.xlsx (1040.6 KB)


## AGEB urbana 2020 (IMU)

In [6]:
# TODO: cargar, filtrar Hermosillo y revisar
# imu = pd.read_csv(RAW_DATA_DIR / "imu_2020.csv")
# imu_hmo = imu[...]
# imu_hmo.head()

In [7]:
# TODO: validar con el esquema pydantic
# from sonora_health_climate_eda.schemas import MarginacionAgebSchema

## Municipal (IMM 2015/2020)

### Paso 1. Leer los tres Excel e inspeccionarlos

`leer_hoja_datos` abre cada libro y elige la hoja que parece contener los datos (la que tiene
una columna de clave municipal). Imprime las hojas y columnas encontradas para que verifiques
que eligió bien. Si el encabezado no está en la primera fila de la hoja, ajusta `HEADER_FILA`.

In [8]:
HEADER_FILA = {anio: 0 for anio in ANIOS_MARGINACION}   # fila donde está el encabezado (0 = primera)


def leer_hoja_datos(ruta: Path, header: int = 0):
    """Devuelve (nombre_hoja, DataFrame, todas_las_hojas) de la hoja con clave municipal."""
    xls = pd.ExcelFile(ruta)
    for hoja in xls.sheet_names:
        df = xls.parse(hoja, header=header)
        columnas = [str(c).strip().upper() for c in df.columns]
        if any("CVE" in c and "MUN" in c for c in columnas):
            return hoja, df, xls.sheet_names
    hoja = xls.sheet_names[0]
    print(f"⚠ No encontré hoja con columna de clave municipal; uso la primera: '{hoja}'")
    return hoja, xls.parse(hoja, header=header), xls.sheet_names


crudos = {}
for anio, ruta in RUTAS_IMM.items():
    hoja, df, hojas = leer_hoja_datos(ruta, header=HEADER_FILA[anio])
    df.columns = [str(c).strip() for c in df.columns]    # quita espacios en los nombres
    crudos[anio] = df

    print("=" * 70)
    print(f"{anio} | hojas: {hojas} | usada: '{hoja}' | {df.shape[0]} filas x {df.shape[1]} columnas")
    print("Columnas:", list(df.columns))
    display(df.head(3))

/var/folders/lm/3h75_s2s0536ztzvnj8lr68m0000gn/T/ipykernel_16155/287937536.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df = xls.parse(hoja, header=header)
/var/folders/lm/3h75_s2s0536ztzvnj8lr68m0000gn/T/ipykernel_16155/287937536.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df = xls.parse(hoja, header=header)


2015 | hojas: ['Base de marginación 2015', 'IMM_2015', 'Diccionario'] | usada: 'IMM_2015' | 2457 filas x 17 columnas
Columnas: ['CVE_ENT', 'NOM_ENT', 'CVE_MUN', 'NOM_MUN', 'POB_TOT', 'ANALF', 'SBASC', 'OVSDE', 'OVSEE', 'OVSAE', 'OVPT', 'VHAC', 'PL.5000', 'PO2SM', 'IM_2015', 'GM_2015', 'IMN_2015']


,CVE_ENT,NOM_ENT,CVE_MUN,NOM_MUN,POB_TOT,ANALF,SBASC,OVSDE,OVSEE,OVSAE,OVPT,VHAC,PL.5000,PO2SM,IM_2015,GM_2015,IMN_2015
0,1,Aguascalientes,1001,Aguascalientes,877190,2.062834,26.303355,0.316501,0.163486,0.722526,0.631526,18.005942,8.731518,31.126336,51.309665,Muy bajo,0.942397
1,1,Aguascalientes,1002,Asientos,46464,4.468127,42.030607,3.863299,1.043886,1.169427,1.687437,30.639918,100.000000,52.761672,47.064347,Bajo,0.864424
2,1,Aguascalientes,1003,Calvillo,56048,4.801101,49.315618,0.596311,0.410663,0.858760,0.932692,30.531817,50.761847,61.952253,47.943584,Bajo,0.880573


/var/folders/lm/3h75_s2s0536ztzvnj8lr68m0000gn/T/ipykernel_16155/287937536.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df = xls.parse(hoja, header=header)
/var/folders/lm/3h75_s2s0536ztzvnj8lr68m0000gn/T/ipykernel_16155/287937536.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df = xls.parse(hoja, header=header)


2020 | hojas: ['Base de marginación 2020', 'IMM_2020', 'Diccionario'] | usada: 'IMM_2020' | 2469 filas x 17 columnas
Columnas: ['CVE_ENT', 'NOM_ENT', 'CVE_MUN', 'NOM_MUN', 'POB_TOT', 'ANALF', 'SBASC', 'OVSDE', 'OVSEE', 'OVSAE', 'OVPT', 'VHAC', 'PL.5000', 'PO2SM', 'IM_2020', 'GM_2020', 'IMN_2020']


,CVE_ENT,NOM_ENT,CVE_MUN,NOM_MUN,POB_TOT,ANALF,SBASC,OVSDE,OVSEE,OVSAE,OVPT,VHAC,PL.5000,PO2SM,IM_2020,GM_2020,IMN_2020
0,1,Aguascalientes,1001,Aguascalientes,948990,1.644738,20.367220,0.104799,0.113169,0.378610,0.591434,10.339530,7.523683,54.226594,60.318795,Muy bajo,0.944508
1,1,Aguascalientes,1002,Asientos,51536,3.526405,33.906364,2.650373,0.486448,0.858160,1.352430,22.942305,78.221049,78.565471,56.546071,Muy bajo,0.885433
2,1,Aguascalientes,1003,Calvillo,58250,4.491509,42.482450,0.365177,0.516760,0.800978,1.040411,19.219858,51.301288,79.259777,57.058251,Muy bajo,0.893453


### Paso 2. Filtrar Sonora

Los archivos son nacionales. Para filtrar se construye una clave de municipio de **5 dígitos**
(entidad + municipio, p. ej. 26030). Si la columna ya trae 5 dígitos se usa tal cual; si trae
solo 3, se combina con la clave de entidad. Filtrar por clave es más seguro que por nombre.

In [9]:
CAND_ENT = ["CVE_ENT", "CVE_EDO", "CLAVE_ENT", "ENT"]
CAND_MUN = ["CVE_MUN", "CVE_MUNI", "CLAVE_MUN", "CVEGEO", "CVE_GEO"]


def a_numero(serie: pd.Series) -> pd.Series:
    """Convierte a numérico: quita espacios y cambia '-' por NaN."""
    if pd.api.types.is_numeric_dtype(serie):
        return serie
    limpia = (
        serie.astype(str)
        .str.replace(r"\s+", "", regex=True)
        .replace({"-": np.nan, "": np.nan, "nan": np.nan, "None": np.nan, "<NA>": np.nan})
    )
    return pd.to_numeric(limpia, errors="coerce")


def buscar_columna(df: pd.DataFrame, candidatas: list) -> str:
    """Devuelve el nombre real de la primera columna que coincida (sin importar mayúsculas)."""
    mapa = {str(c).strip().upper(): c for c in df.columns}
    for cand in candidatas:
        if cand in mapa:
            return mapa[cand]
    raise KeyError(f"No encontré ninguna de {candidatas}. Columnas disponibles: {list(df.columns)}")


def clave_5d(df: pd.DataFrame) -> pd.Series:
    """Clave de municipio de 5 dígitos (entidad*1000 + municipio)."""
    mun = a_numero(df[buscar_columna(df, CAND_MUN)])
    if mun.max() >= 1000:                    # ya viene la clave completa
        return mun
    ent = a_numero(df[buscar_columna(df, CAND_ENT)])
    return ent * 1000 + mun


sonora = {}
for anio, df in crudos.items():
    df = df.copy()
    df["CVE_MUN_5D"] = clave_5d(df).astype("Int64")
    mascara = (df["CVE_MUN_5D"] // 1000 == CVE_ENT_SONORA).fillna(False)
    sonora[anio] = df[mascara].copy()        # .copy() evita advertencias al modificar luego
    print(f"{anio}: {len(df)} filas nacionales -> {len(sonora[anio])} de Sonora")

    if sonora[anio].empty:
        raise ValueError(f"{anio}: el filtro de Sonora quedó vacío. Revisa CAND_ENT / CAND_MUN.")

2015: 2457 filas nacionales -> 72 de Sonora
2020: 2469 filas nacionales -> 72 de Sonora


### Paso 3. Limpiar tipos y unificar nombres de columnas

- **Tipos:** cada columna de texto se intenta convertir a número, y **solo se convierte si no se pierde ningún dato real** (los `-` cuentan como vacío). Así los nombres y los grados de marginación se quedan como texto, y los indicadores pasan a numéricos sin riesgo de borrar datos.
- **Nombres:** se quita el sufijo de año (`IM_2020` → `IM`, `GM_2020` → `GM`) para que los tres años compartan nombres. Se agrega la columna `AÑO`.

In [10]:
def convertir_si_es_numerica(serie: pd.Series) -> pd.Series:
    """Convierte a número solo si no se pierde ningún valor real; si no, deja la columna igual."""
    if pd.api.types.is_numeric_dtype(serie):
        return serie
    s = serie.astype("string").str.strip().str.replace(r"\s+", "", regex=True)
    s = s.replace({"-": pd.NA, "": pd.NA})
    num = pd.to_numeric(s, errors="coerce")
    if num.notna().sum() == s.notna().sum():    # no se perdió nada: era una columna numérica
        return num
    return serie                                # tiene texto real (nombres, grados): se conserva


def quitar_sufijo_anio(nombre: str) -> str:
    """'IM_2020' -> 'IM', 'GM 2015' -> 'GM'. No toca nombres sin sufijo de año."""
    return re.sub(r"[_\s.]?(2010|2015|2020)$", "", str(nombre).strip())


unificados = {}
for anio, df in sonora.items():
    df = df.copy()
    for col in df.columns:
        df[col] = convertir_si_es_numerica(df[col])

    df.columns = [quitar_sufijo_anio(c) for c in df.columns]
    assert not pd.Index(df.columns).duplicated().any(), f"{anio}: columnas duplicadas tras quitar el sufijo"

    df["AÑO"] = anio
    primeras = ["AÑO", "CVE_MUN_5D"]
    unificados[anio] = df[primeras + [c for c in df.columns if c not in primeras]].reset_index(drop=True)

for anio, df in unificados.items():
    print(anio, "->", list(df.columns))

2015 -> ['AÑO', 'CVE_MUN_5D', 'CVE_ENT', 'NOM_ENT', 'CVE_MUN', 'NOM_MUN', 'POB_TOT', 'ANALF', 'SBASC', 'OVSDE', 'OVSEE', 'OVSAE', 'OVPT', 'VHAC', 'PL.5000', 'PO2SM', 'IM', 'GM', 'IMN']
2020 -> ['AÑO', 'CVE_MUN_5D', 'CVE_ENT', 'NOM_ENT', 'CVE_MUN', 'NOM_MUN', 'POB_TOT', 'ANALF', 'SBASC', 'OVSDE', 'OVSEE', 'OVSAE', 'OVPT', 'VHAC', 'PL.5000', 'PO2SM', 'IM', 'GM', 'IMN']


### Paso 4. Validar

- Un solo registro de Hermosillo (26030) por año y sin claves repetidas (si falla, el `assert` detiene la ejecución).
- Número de municipios de Sonora por año. Se espera 72 en 2020; en 2015 puede variar, por eso solo avisa.
- Matriz de datos no nulos por columna y año: un 0 indica una variable que no existe en ese año.
- Fila de Hermosillo en cada año, para comprobar que las escalas sean consistentes.

In [11]:
for anio, df in unificados.items():
    assert df["CVE_MUN_5D"].is_unique, f"{anio}: hay claves de municipio repetidas"
    assert (df["CVE_MUN_5D"] == CVE_MUN_HERMOSILLO).sum() == 1, f"{anio}: Hermosillo no aparece exactamente una vez"
    if len(df) != 72:
        print(f"⚠ {anio}: {len(df)} municipios de Sonora (se esperaban 72). Revisa si es correcto para ese año.")

# Vista comparativa en memoria (no se guarda): todos los años juntos
todos = pd.concat(unificados.values(), ignore_index=True, sort=False)

print("Valores no nulos por columna y año (0 = la variable no existe ese año):")
display(todos.groupby("AÑO").count().T)

print("\nHermosillo en cada año:")
display(todos[todos["CVE_MUN_5D"] == CVE_MUN_HERMOSILLO].set_index("AÑO").T)

if "IM" in todos.columns:
    print("\nDistribución del IM por año (¿misma escala entre años?):")
    display(todos.groupby("AÑO")["IM"].describe())

Valores no nulos por columna y año (0 = la variable no existe ese año):


AÑO,2015,2020
CVE_MUN_5D,72,72
CVE_ENT,72,72
NOM_ENT,72,72
CVE_MUN,72,72
NOM_MUN,72,72
POB_TOT,72,72
ANALF,72,72
SBASC,72,72
OVSDE,72,72
OVSEE,72,72



Hermosillo en cada año:


AÑO,2015,2020
CVE_MUN_5D,26030,26030
CVE_ENT,26,26
NOM_ENT,Sonora,Sonora
CVE_MUN,26030,26030
NOM_MUN,Hermosillo,Hermosillo
POB_TOT,884273,936263
ANALF,1.292487,1.43976
SBASC,20.613987,17.562774
OVSDE,0.571695,0.187077
OVSEE,0.398693,0.258912



Distribución del IM por año (¿misma escala entre años?):


,count,mean,std,min,25%,50%,75%,max
AÑO,,,,,,,,
2015,72.0,48.464244,1.981256,39.450331,47.521181,48.788664,49.738501,51.891868
2020,72.0,57.101684,1.728517,49.441085,56.406890,57.225844,58.150784,60.443754


In [12]:
# Un solo DataFrame de Hermosillo: una fila por año, variables como columnas
hermosillo = (
    todos[todos["CVE_MUN_5D"] == CVE_MUN_HERMOSILLO]   # solo Hermosillo (26030)
    .sort_values("AÑO")                                # ordena por año
    .reset_index(drop=True)                            # reinicia el índice 0, 1, 2
)

display(hermosillo)
print(hermosillo.shape)    # esperado: (2, 19)

,AÑO,CVE_MUN_5D,CVE_ENT,NOM_ENT,CVE_MUN,NOM_MUN,POB_TOT,ANALF,SBASC,OVSDE,OVSEE,OVSAE,OVPT,VHAC,PL.5000,PO2SM,IM,GM,IMN
0,2015,26030,26,Sonora,26030,Hermosillo,884273,1.292487,20.613987,0.571695,0.398693,1.576824,1.648960,23.297346,4.125988,19.951522,51.512968,Muy bajo,0.946131
1,2020,26030,26,Sonora,26030,Hermosillo,936263,1.439760,17.562774,0.187077,0.258912,0.488080,1.791786,13.896215,3.713914,47.174923,60.443754,Muy bajo,0.946465


(2, 19)


## Cartografía AGEB (INEGI)

In [13]:
# TODO: descargar el MGN 2020 y recortar las AGEBs de Hermosillo (geopandas)

## Guardar salidas

In [14]:
# Columnas auxiliares que agregamos en el notebook; panel.py las trataría como
# indicadores, así que se quitan de los CSV por año (el año ya va en el nombre del archivo)
COLUMNAS_AUXILIARES = ["AÑO", "CVE_MUN_5D"]

# 1) Un CSV por año con los municipios de Sonora (los lee panel.py)
for anio, df in unificados.items():
    ruta = MARGINACION_INTERIM_DIR / f"marginacion_municipal_{anio}.csv"
    df.drop(columns=COLUMNAS_AUXILIARES).to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"Guardado: {ruta.name} ({len(df)} filas)")

# 2) Un CSV con solo Hermosillo, una fila por año (conserva AÑO)
ruta_hermosillo = MARGINACION_INTERIM_DIR / f"marginacion_municipal_hermosillo_{ANIOS_MARGINACION[0]}_{ANIOS_MARGINACION[-1]}.csv"
hermosillo.to_csv(ruta_hermosillo, index=False, encoding="utf-8-sig")
print(f"Guardado: {ruta_hermosillo.name} ({len(hermosillo)} filas)")
# TODO: guardar también marginacion_imu_2020.csv cuando se complete la sección AGEB

Guardado: marginacion_municipal_2015.csv (72 filas)
Guardado: marginacion_municipal_2020.csv (72 filas)
Guardado: marginacion_municipal_hermosillo_2015_2020.csv (2 filas)


In [15]:
from sonora_health_climate_eda.panel import marginacion_hermosillo

margin = marginacion_hermosillo()
print(margin.shape)             # esperado: 72 filas (ene-2015 a dic-2020, un mes por fila)
print(list(margin.columns))     # debe empezar con fecha_mes y NO incluir AÑO ni CVE_MUN_5D
display(margin.head(3))

2026-11-13 15:44:58.833 -07:00 | INFO     | sonora_health_climate_eda.panel:marginacion_hermosillo:414 - Marginación interpolada: 72 meses, 11 indicadores: ['ANALF', 'SBASC', 'OVSDE', 'OVSEE', 'OVSAE', 'OVPT', 'VHAC', 'PL.5000', 'PO2SM', 'IM', 'IMN']


(72, 12)
['fecha_mes', 'ANALF', 'SBASC', 'OVSDE', 'OVSEE', 'OVSAE', 'OVPT', 'VHAC', 'PL.5000', 'PO2SM', 'IM', 'IMN']


,fecha_mes,ANALF,SBASC,OVSDE,OVSEE,OVSAE,OVPT,VHAC,PL.5000,PO2SM,IM,IMN
0,2015-01-01,1.292487,20.613987,0.571695,0.398693,1.576824,1.648960,23.297346,4.125988,19.951522,51.512968,0.946131
1,2015-02-01,1.294988,20.562186,0.565166,0.396320,1.558341,1.651385,23.137743,4.118992,20.413693,51.664586,0.946137
2,2015-03-01,1.297246,20.515399,0.559268,0.394176,1.541646,1.653575,22.993586,4.112674,20.831139,51.801532,0.946142
